<a href="https://colab.research.google.com/github/Abdulkadir03/Crop-Recommender-for-Sahel-Region/blob/main/notebooks/07_training_table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07: Training table

**Purpose:** Join all feature files and the SPAM labels into one table with one row per grid cell, and define the pixel weights and spatial cross-validation folds used for modelling.

**Inputs** (notebooks 01-05): `grid`, `climate`, `soil_terrain`, `salinity`, `sentinel`, `labels_pixel` and `grid_pixel_map` parquet files.

**Method:**
1. Join the feature files and the pixel labels on `grid_id` (and SPAM pixel index).
2. Group the features: climate (including aridity index), soil, terrain, salinity, sentinel and modis. Coordinates, country and cropland fraction are kept as metadata and are not model features.
3. Weight each labelled cell by 1 / (number of grid cells in its SPAM pixel), so each pixel counts once.
4. Assign spatial folds: 100 km blocks (located by SPAM pixel centre for labelled cells), distributed over 5 folds balanced by number of pixels. All cells of a pixel stay in one fold.

**Outputs** (Google Drive): `training_table.parquet` (one row per grid cell); `training_config.json` (feature list, groups, crops, fold settings; small enough to commit).

**Notes:**
- Satellite vegetation features (Sentinel, MODIS) partly reflect what is already grown (for example irrigated rice), so models are compared with and without them.
- Climate sources are coarse (4 to 11 km), so folds use blocks much larger than that.
- Cells without a SPAM pixel (0.26%) have no label and weight 0.
- 81 features in six groups: climate 23 (including aridity index), soil 18, terrain 5, salinity 5, sentinel 26, modis 4.
- 300,365 of 301,142 cells have a label; there are 21,128 labelled pixels and the weights sum to 21,128.
- 326 blocks of 100 km in 5 folds of about 4,225 labelled pixels each. The country mix differs between folds (for example Nigeria is 13% to 34% of cells), so per-fold scores vary.
- 35 feature pairs have |r| > 0.95 (for example sal_mean_2016 and sal_share_2016); they are kept here and pruned in later notebooks.

In [ ]:
# @title Cell 1: load and check
import os, json
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive'
P = f'{BASE}/crop_project'

grid    = pd.read_parquet(f'{P}/grid.parquet')
climate = pd.read_parquet(f'{P}/climate.parquet')
soil    = pd.read_parquet(f'{P}/soil_terrain.parquet')
salin   = pd.read_parquet(f'{P}/salinity.parquet')
senti   = pd.read_parquet(f'{P}/sentinel.parquet')
labels  = pd.read_parquet(f'{P}/labels_pixel.parquet')
gmap    = pd.read_parquet(f'{P}/grid_pixel_map.parquet')

ref = set(grid.grid_id)
for name, d in [('climate', climate), ('soil_terrain', soil), ('salinity', salin),
                ('sentinel', senti), ('grid_pixel_map', gmap)]:
    print(f'{name:15} rows={len(d):>7} unique={d.grid_id.nunique():>7} same ids as grid={set(d.grid_id) == ref}')
print('labels_pixel:', labels.shape)

In [ ]:
# @title Cell 2: build the table
CROPS = ['sorg', 'pmil', 'cowp', 'grou', 'maiz', 'rice', 'cass', 'cott',
         'yams', 'sesa', 'onio', 'soyb', 'toma', 'bean', 'swpo', 'sugc']

groups = {
    'climate':  [c for c in climate.columns if c != 'grid_id'] + ['aridity_index'],
    'soil':     [c for c in soil.columns if c.endswith(('_top', '_sub'))],
    'terrain':  [c for c in soil.columns if c in ('elevation', 'slope_deg', 'northness', 'eastness', 'elevation_std')],
    'salinity': [c for c in salin.columns if c != 'grid_id'],
    'modis':    [c for c in senti.columns if c.startswith('modis_')],
    'sentinel': [c for c in senti.columns if c != 'grid_id' and not c.startswith('modis_')]}
FEATURES = sum(groups.values(), [])
allfeat = (set(climate.columns) | set(soil.columns) | set(salin.columns) | set(senti.columns)) - {'grid_id'}
print({k: len(v) for k, v in groups.items()}, '| total', len(FEATURES))
print('Columns not assigned to a group:', allfeat - set(FEATURES))

meta = (grid[['grid_id', 'lon', 'lat', 'adm0', 'adm1', 'crop_frac', 'aridity_index']]
        .merge(gmap[['grid_id', 'px', 'py', 'has_label']], on='grid_id', how='left'))
tab = (meta.merge(climate, on='grid_id', how='left').merge(soil, on='grid_id', how='left')
           .merge(salin, on='grid_id', how='left').merge(senti, on='grid_id', how='left'))

lab_cols = (['px', 'py', 'total_area', 'irrig_share', 'n_cells']
            + [f'share_{c}' for c in CROPS] + [f'present_{c}' for c in CROPS])
tab = tab.merge(labels[lab_cols], on=['px', 'py'], how='left')
tab['has_label'] = tab['has_label'].fillna(False).astype(bool) & tab['n_cells'].notna()
tab['weight'] = np.where(tab.has_label, 1.0 / tab.n_cells, 0.0)
tab[FEATURES] = tab[FEATURES].astype('float32')
for c in CROPS:
    tab[f'present_{c}'] = tab[f'present_{c}'].astype('float32')

cnt = tab[tab.has_label].groupby(['px', 'py']).size().rename('cnt').reset_index()
chk = labels[['px', 'py', 'n_cells']].merge(cnt, on=['px', 'py'])
print('\nn_cells matches the number of grid cells per pixel:', bool((chk.n_cells == chk.cnt).all()))
print('Rows:', len(tab), '| labelled cells:', int(tab.has_label.sum()),
      f'| sum of weights: {tab.weight.sum():.0f} (should equal the {len(labels)} pixels)')
print('Features with missing values (%):')
nan = (tab[FEATURES].isna().mean() * 100).round(2)
print(nan[nan > 0].sort_values(ascending=False))

In [ ]:
# @title Cell 3: spatial folds
import geopandas as gpd

BLOCK_KM, N_FOLDS, SEED = 100, 5, 42

# Block location: SPAM pixel centre for labelled cells (keeps a pixel inside one block), own location otherwise
lon_c = np.where(tab.has_label, (tab.px.astype(float) + 0.5) / 12 - 180, tab.lon)
lat_c = np.where(tab.has_label, 90 - (tab.py.astype(float) + 0.5) / 12, tab.lat)
xy = gpd.GeoSeries(gpd.points_from_xy(lon_c, lat_c), crs='EPSG:4326').to_crs('EPSG:6933')
tab['block'] = (np.floor(xy.x / (BLOCK_KM * 1000)).astype(int).astype(str) + '_' +
                np.floor(xy.y / (BLOCK_KM * 1000)).astype(int).astype(str))

size = tab.groupby('block').agg(w=('weight', 'sum'), n=('grid_id', 'size'))
size['key'] = size.w + 1e-6 * size.n
order = size.sample(frac=1, random_state=SEED).sort_values('key', ascending=False, kind='stable').index
load, fold_of = np.zeros(N_FOLDS), {}
for b in order:
    k = int(np.argmin(load)); fold_of[b] = k; load[k] += size.loc[b, 'key']
tab['fold'] = tab.block.map(fold_of).astype('int8')

per_pix = tab[tab.has_label].groupby(['px', 'py']).fold.nunique()
print('Blocks:', len(size), '| every pixel in a single fold:', bool((per_pix == 1).all()))
print('\nPer fold:')
print(tab.groupby('fold').agg(cells=('grid_id', 'size'), labelled_pixels=('weight', 'sum')).round(0))
print('\nShare of each fold\'s cells by country (%):')
print((pd.crosstab(tab.fold, tab.adm0, normalize='index') * 100).round(1))
print('\nMean crop share per fold (labelled cells), to check the folds are similar:')
print(tab[tab.has_label].groupby('fold')[[f'share_{c}' for c in CROPS]].mean().round(3).T)

In [ ]:
# @title Cell 4: correlations and saving
smp = tab[FEATURES].sample(30000, random_state=0)
cm = smp.corr().abs()
iu = np.triu_indices_from(cm, k=1)
pairs = pd.DataFrame({'a': cm.index[iu[0]], 'b': cm.columns[iu[1]], 'r': cm.values[iu]}).dropna()
print('Feature pairs with |r| > 0.95:', int((pairs.r > 0.95).sum()), 'of', len(pairs))
print(pairs.sort_values('r', ascending=False).head(12).round(3).to_string(index=False))
print('Constant features:', [c for c in FEATURES if smp[c].nunique() <= 1])

meta_cols = ['grid_id', 'lon', 'lat', 'adm0', 'adm1', 'crop_frac', 'px', 'py', 'has_label',
             'n_cells', 'weight', 'block', 'fold', 'total_area', 'irrig_share']
lab_out = [f'share_{c}' for c in CROPS] + [f'present_{c}' for c in CROPS]
out = tab[meta_cols + FEATURES + lab_out]
out.to_parquet(f'{P}/training_table.parquet', index=False)
with open(f'{P}/training_config.json', 'w') as f:
    json.dump({'features': FEATURES, 'groups': groups, 'crops': CROPS,
               'block_km': BLOCK_KM, 'n_folds': N_FOLDS, 'seed': SEED}, f, indent=1)
print('Saved training_table.parquet', out.shape, f'({os.path.getsize(f"{P}/training_table.parquet") / 1e6:.0f} MB)')
drive.flush_and_unmount(); drive.mount('/content/drive')